### **Student Information**
GitHub ID: huichieh1123

Kaggle name: huichieh

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

## 1. Model Development

This report focuses on `submission_blend_5fold.csv`, a five-fold probability ensemble of Twitter-RoBERTa and DeBERTa. The Competition Code section explains the preprocessing, feature representation, training, and leaderboard results.

### 1.1 Preprocessing Steps

* Loaded `final_posts.json`, `data_identification.csv`, `emotion.csv`, and `sampleSubmission.csv`, then joined them by `post_id` with one-to-one validation.
* Split the merged data according to the provided `set` column, producing 3,360 training examples and 1,440 test examples.
* Applied Twitter-RoBERTa-compatible normalization: HTML entities were decoded, URLs were mapped to `http`, mentions were mapped to `@user`, and repeated whitespace was collapsed. Emoji, hashtags, punctuation, and letter case were retained because they contain emotion information.
* Used `StratifiedGroupKFold` with normalized text as the group. Consequently, duplicate normalized posts cannot occur in both the training and validation portions of a fold.

### 1.2 Feature Engineering Steps

The model tokenizer provides subword input IDs and attention masks. Sequences were truncated to 128 tokens and dynamically padded per batch. No manual TF-IDF or lexicon features were added. The principal task-specific features came from the pretrained Twitter-RoBERTa representations. To handle class imbalance, each fold used inverse-frequency class weights, $w_c=N/(K n_c)$, in cross-entropy loss.

The source checkpoint has an eleven-emotion output layer. Instead of discarding it, the six rows corresponding to `anger`, `disgust`, `fear`, `joy`, `sadness`, and `surprise` were copied into the six-class output layer. This preserves useful emotion-specific initialization.

### 1.3 Explanation of the Model

Each fold fine-tuned Twitter-RoBERTa-large with learning rate `1e-5`, effective batch size 32, weight decay 0.01, 10% warm-up, gradient clipping, gradient checkpointing, and at most eight epochs. Macro-F1 selected the best checkpoint, and early stopping used patience 2. The five models produced OOF predictions for evaluation and their test logits were averaged. For exact normalized train/test duplicates, the known training label was used after ensembling. The final CSV was reordered through `sampleSubmission.csv` and checked for 1,440 non-null predictions.

### 1.4 AI Prompts and Decisions

The main prompts used during development were: (1) inspect the five-fold Twitter-RoBERTa result (2) try DeBERTa and blend its probabilities (3) try different DeBERTa learning rates (4) build an anger-versus-disgust second-stage classifier. 

AI assistance was used to structure controlled OOF comparisons and fix experiment code I wrote.

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

We join post text, split assignments, and emotion labels by `post_id`, checking that each join is one-to-one. The supplied split gives 3,360 labeled training posts and 1,440 test posts. We decode HTML entities, replace URLs with `http` and user mentions with `@user`, and collapse whitespace. This reduces variation from URLs and usernames while preserving potentially informative signals such as emoji, hashtags, capitalization, and punctuation.

For validation, we form five stratified folds and use normalized text as the group key. All copies of the same normalized text stay in one fold, which prevents duplicates from inflating OOF scores through overlap between a fold's training and validation sets.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import html
import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, EarlyStoppingCallback, Trainer, TrainingArguments,
)

SEED = 42
MODEL_NAME = "cardiffnlp/twitter-roberta-large-emotion-latest"
MAX_LENGTH = 128
N_FOLDS = 5
MAX_EPOCHS = 8
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 16
GRAD_ACCUMULATION = 8  # effective batch size = 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

default_data_dir = Path("dataset") if Path("dataset").is_dir() else Path("../dataset")
DATA_DIR = Path(os.environ.get("LAB2_DATA_DIR", default_data_dir))
assert DATA_DIR.exists(), f"Dataset directory not found: {DATA_DIR.resolve()}"
with (DATA_DIR / "final_posts.json").open(encoding="utf-8") as file:
    posts = pd.DataFrame(json.load(file))
identification = pd.read_csv(DATA_DIR / "data_identification.csv")
emotion = pd.read_csv(DATA_DIR / "emotion.csv")
sample_submission = pd.read_csv(DATA_DIR / "sampleSubmission.csv")

data = posts.merge(identification, on="post_id", validate="one_to_one")
data = data.merge(emotion, on="post_id", how="left", validate="one_to_one")
train_df = data.loc[data["set"].eq("train")].copy().reset_index(drop=True)
test_df = data.loc[data["set"].eq("test")].copy().reset_index(drop=True)

def normalize_tweet(text):
    text = html.unescape(str(text))
    text = re.sub(r"(?:https?://|www\.)\S+", "http", text, flags=re.I)
    text = re.sub(r"@(?:<username>|[A-Za-z0-9_]+)", "@user", text, flags=re.I)
    return re.sub(r"\s+", " ", text).strip()

train_df["clean_text"] = train_df["text"].map(normalize_tweet)
test_df["clean_text"] = test_df["text"].map(normalize_tweet)
labels = sorted(train_df["label"].unique())
label2id = {label: index for index, label in enumerate(labels)}
id2label = {index: label for label, index in label2id.items()}
train_df["label_id"] = train_df["label"].map(label2id)

assert len(train_df) == 3360 and len(test_df) == 1440
assert train_df["label"].notna().all() and test_df["label"].isna().all()
assert set(sample_submission["post_id"]) == set(test_df["post_id"])
assert not train_df["clean_text"].eq("").any()

splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(splitter.split(
    train_df["clean_text"], train_df["label"], groups=train_df["clean_text"]
))
text_to_label = train_df.groupby("clean_text")["label"].agg(
    lambda values: sorted(values.mode().tolist())[0]
)
print(f"train={len(train_df)}, test={len(test_df)}, labels={labels}")
display(train_df["label"].value_counts().reindex(labels).rename("count").to_frame())

## 2. Feature Engineering Steps

Each checkpoint's tokenizer converts text into subword token IDs and attention masks. We truncate to 128 tokens and pad only to the longest sequence in each batch. The transformer learns contextual features directly from these inputs. We compute inverse-frequency class weights using only the training portion of each fold and apply them in cross-entropy so that frequent labels do not dominate the loss.

We choose `cardiffnlp/twitter-roberta-large-emotion-latest` because its Twitter-domain representations and existing emotion classifier fit the task. We initialize the new six-class output layer by copying the weights and biases for the six matching emotions from its 11-class head. For the separate blend experiment, we use `microsoft/deberta-v3-large` as a general-domain checkpoint with a newly initialized six-class head. Its different pretraining gives the two models a chance to make complementary errors.

In [ ]:
def tokenize_frame(frame, tokenizer, with_labels=True):
    columns = ["clean_text", "label_id"] if with_labels else ["clean_text"]
    dataset = Dataset.from_pandas(frame[columns], preserve_index=False)
    if with_labels:
        dataset = dataset.rename_column("label_id", "labels")
    return dataset.map(
        lambda batch: tokenizer(
            batch["clean_text"], truncation=True, max_length=MAX_LENGTH
        ),
        batched=True,
        remove_columns=["clean_text"],
    )

def balanced_weights(label_ids):
    ids = np.asarray(label_ids, dtype=int)
    counts = np.bincount(ids, minlength=len(labels))
    return torch.tensor(len(ids) / (len(labels) * counts), dtype=torch.float32)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        target = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, target,
            weight=self.class_weights.to(outputs.logits.device),
        )
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_prediction):
    logits, target = eval_prediction
    prediction = logits.argmax(axis=-1)
    return {
        "macro_f1": f1_score(target, prediction, average="macro"),
        "weighted_f1": f1_score(target, prediction, average="weighted"),
        "accuracy": accuracy_score(target, prediction),
    }

def task_adapted_model():
    # Copy the six matching emotion rows from the pretrained 11-label head.
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    source_label2id = {
        label.lower(): int(index) for label, index in model.config.label2id.items()
    }
    missing = set(labels) - set(source_label2id)
    assert not missing, f"Source head is missing labels: {sorted(missing)}"
    old_output = model.classifier.out_proj
    new_output = torch.nn.Linear(old_output.in_features, len(labels))
    with torch.no_grad():
        for target_index, label in id2label.items():
            source_index = source_label2id[label]
            new_output.weight[target_index].copy_(old_output.weight[source_index])
            new_output.bias[target_index].copy_(old_output.bias[source_index])
    model.classifier.out_proj = new_output
    model.num_labels = len(labels)
    model.config.num_labels = len(labels)
    model.config.label2id = label2id
    model.config.id2label = id2label
    return model

def make_arguments(output_dir, seed, eval_batch_size=EVAL_BATCH_SIZE):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=1e-5,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=eval_batch_size,
        gradient_accumulation_steps=GRAD_ACCUMULATION,
        num_train_epochs=MAX_EPOCHS,
        weight_decay=0.01,
        warmup_ratio=0.10,
        max_grad_norm=1.0,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        bf16=USE_BF16,
        fp16=USE_FP16,
        report_to="none",
        seed=seed,
        data_seed=seed,
    )

## 3. Model Implementation Steps

The cell below fine-tunes one Twitter-RoBERTa model per fold, selects the best checkpoint by validation macro-F1, and collects predictions for each held-out fold to compute the aggregate OOF score. It then averages the five test logits and selects the highest-scoring class. For exact matches to normalized training text, the submission uses the most frequent training label for that text. Finally, predictions are joined to the sample submission by ID and exported as `submission_twitter_roberta_emotion_5fold.csv`.

For the blend, we fine-tune DeBERTa on the same five folds. The original OOF weight search selected 62.5% Twitter-RoBERTa and 37.5% DeBERTa. The cells below combine the two model families' probabilities and apply the exact-match rule before writing `submission_blend_5fold.csv`.

In [ ]:
# This cell trains five large models and therefore requires a CUDA GPU.
RUN_FULL_5_FOLD = False  # Enable only to train Twitter-RoBERTa from scratch.
PREDICTION_DIR = Path(os.environ.get("LAB2_PREDICTION_DIR", "deberta_blend_experiment"))

if RUN_FULL_5_FOLD:
    PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    full_train_tokens = tokenize_frame(train_df, tokenizer)
    test_tokens = tokenize_frame(test_df, tokenizer, with_labels=False)
    oof_logits = np.zeros((len(train_df), len(labels)), dtype=np.float32)
    test_logits = np.zeros((len(test_df), len(labels)), dtype=np.float32)
    fold_scores = []

    for fold, (fold_train_indices, fold_valid_indices) in enumerate(splits):
        print(f"\n===== Twitter-RoBERTa fold {fold + 1}/{N_FOLDS} =====")
        weights = balanced_weights(train_df.iloc[fold_train_indices]["label_id"])
        trainer = WeightedTrainer(
            model=task_adapted_model(),
            args=make_arguments(
                f"twitter_roberta_emotion_fold_{fold}", SEED + fold
            ),
            train_dataset=full_train_tokens.select(fold_train_indices),
            eval_dataset=full_train_tokens.select(fold_valid_indices),
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            class_weights=weights,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        )
        trainer.train()
        valid_logits = trainer.predict(
            full_train_tokens.select(fold_valid_indices)
        ).predictions
        oof_logits[fold_valid_indices] = valid_logits
        fold_score = f1_score(
            train_df.iloc[fold_valid_indices]["label_id"],
            valid_logits.argmax(axis=1),
            average="macro",
        )
        fold_scores.append(fold_score)
        print(f"fold {fold + 1} macro-F1: {fold_score:.6f}")
        fold_test_logits = trainer.predict(test_tokens).predictions
        test_logits += fold_test_logits / N_FOLDS
        np.save(PREDICTION_DIR / f"twitter_fold{fold}_valid_logits.npy", valid_logits)
        np.save(PREDICTION_DIR / f"twitter_fold{fold}_test_logits.npy", fold_test_logits)
        del trainer
        gc.collect()
        torch.cuda.empty_cache()

    oof_prediction = oof_logits.argmax(axis=1)
    oof_macro_f1 = f1_score(
        train_df["label_id"], oof_prediction, average="macro"
    )
    print("fold scores:", [round(score, 6) for score in fold_scores])
    print(f"5-fold OOF macro-F1: {oof_macro_f1:.6f}")
    print(classification_report(
        train_df["label_id"],
        oof_prediction,
        labels=list(range(len(labels))),
        target_names=labels,
        digits=4,
    ))

    # Average the five test logits, then apply the duplicate-text rule used
    # by the best Kaggle submission.
    test_prediction = np.array(
        [id2label[index] for index in test_logits.argmax(axis=1)], dtype=object
    )
    known_labels = test_df["clean_text"].map(text_to_label)
    test_prediction[known_labels.notna().to_numpy()] = known_labels.dropna().to_numpy()
    prediction_frame = pd.DataFrame({
        "post_id": test_df["post_id"],
        "label": test_prediction,
    })
    submission = sample_submission[["post_id"]].merge(
        prediction_frame, on="post_id", how="left", validate="one_to_one"
    ).rename(columns={"post_id": "id"})
    assert submission.columns.tolist() == ["id", "label"]
    assert len(submission) == 1440
    assert submission["id"].is_unique and submission["label"].notna().all()
    assert set(submission["label"]).issubset(set(labels))
    submission.to_csv(
        "submission_twitter_roberta_emotion_5fold.csv", index=False
    )
    display(submission.head())
    print("Saved submission_twitter_roberta_emotion_5fold.csv")
else:
    print("Five-fold training skipped. Set RUN_FULL_5_FOLD = True to run.")

### 3.1 DeBERTa five-fold training

DeBERTa uses the same fold indices and fold-local class weighting as Twitter-RoBERTa, with its own tokenizer and a new six-class head. Each fold saves its validation indices and the raw logits for validation and test. Leave the training switches off to reuse existing predictions. Set `LAB2_PREDICTION_DIR` to the existing `deberta_blend_experiment` directory before running the notebook.

In [ ]:
RUN_DEBERTA_5_FOLD = False
DEBERTA_MODEL_NAME = "microsoft/deberta-v3-large"

if RUN_DEBERTA_5_FOLD:
    deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL_NAME, use_fast=True)
    deberta_collator = DataCollatorWithPadding(tokenizer=deberta_tokenizer)
    deberta_train = tokenize_frame(train_df, deberta_tokenizer)
    deberta_test = tokenize_frame(test_df, deberta_tokenizer, with_labels=False)

    for fold, (fit_indices, valid_indices) in enumerate(splits):
        fold_dir = PREDICTION_DIR / f"fold_{fold}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        # The original runs initialized each fold from the same checkpoint and seed.
        torch.manual_seed(SEED)
        model = AutoModelForSequenceClassification.from_pretrained(
            DEBERTA_MODEL_NAME,
            num_labels=len(labels),
            label2id=label2id,
            id2label=id2label,
            ignore_mismatched_sizes=True,
        )
        valid_tokens = deberta_train.select(valid_indices)
        trainer = WeightedTrainer(
            model=model,
            args=make_arguments(str(fold_dir), SEED + fold, eval_batch_size=32),
            train_dataset=deberta_train.select(fit_indices),
            eval_dataset=valid_tokens,
            data_collator=deberta_collator,
            compute_metrics=compute_metrics,
            class_weights=balanced_weights(train_df.iloc[fit_indices]["label_id"]),
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        )
        trainer.train()
        np.save(fold_dir / "valid_indices.npy", valid_indices)
        np.save(fold_dir / "valid_logits.npy", trainer.predict(valid_tokens).predictions)
        np.save(fold_dir / "test_logits.npy", trainer.predict(deberta_test).predictions)
        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()
else:
    print("DeBERTa training skipped. The blend uses saved fold predictions.")

### 3.2 Blend probabilities and export the Kaggle submission

We convert each fold's logits to probabilities, place validation probabilities back in the original training row order, and average test probabilities across folds for each model. We use the recorded weight of 0.625 for Twitter-RoBERTa and 0.375 for DeBERTa. OOF macro-F1 is computed before the test-only exact-match rule. The export uses the sample submission's ID order. Cached predictions must come from the same dataset, row order, label mapping, and folds.

In [ ]:
def load_probabilities(path, expected_rows):
    logits = np.load(path, allow_pickle=False).astype(np.float32)
    if logits.shape != (expected_rows, len(labels)) or not np.isfinite(logits).all():
        raise ValueError(f"Invalid logits in {path}: {logits.shape}")
    exp_logits = np.exp(logits - logits.max(axis=1, keepdims=True))
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def export_blend(prediction_dir):
    twitter_oof = np.empty((len(train_df), len(labels)), dtype=np.float32)
    deberta_oof = np.empty_like(twitter_oof)
    twitter_test, deberta_test = [], []

    for fold, (_, valid_indices) in enumerate(splits):
        fold_dir = prediction_dir / f"fold_{fold}"
        saved_indices = np.load(fold_dir / "valid_indices.npy", allow_pickle=False)
        if not np.array_equal(saved_indices, valid_indices):
            raise ValueError(f"Saved DeBERTa validation indices differ for fold {fold}.")
        twitter_oof[valid_indices] = load_probabilities(
            prediction_dir / f"twitter_fold{fold}_valid_logits.npy", len(valid_indices)
        )
        deberta_oof[valid_indices] = load_probabilities(
            fold_dir / "valid_logits.npy", len(valid_indices)
        )
        twitter_test.append(load_probabilities(
            prediction_dir / f"twitter_fold{fold}_test_logits.npy", len(test_df)
        ))
        deberta_test.append(load_probabilities(fold_dir / "test_logits.npy", len(test_df)))

    twitter_weight = 0.625  # Selected in the original OOF weight search.
    deberta_weight = 1 - twitter_weight
    blend_oof = twitter_weight * twitter_oof + deberta_weight * deberta_oof
    blend_score = f1_score(train_df["label_id"], blend_oof.argmax(axis=1), average="macro")
    print(f"Blend OOF macro-F1: {blend_score:.6f}")

    blend_test = (
        twitter_weight * np.mean(twitter_test, axis=0)
        + deberta_weight * np.mean(deberta_test, axis=0)
    )
    prediction = pd.Series([id2label[i] for i in blend_test.argmax(axis=1)])
    prediction = test_df["clean_text"].map(text_to_label).combine_first(prediction)
    prediction_frame = pd.DataFrame({"post_id": test_df["post_id"], "label": prediction})
    submission = sample_submission[["post_id"]].merge(
        prediction_frame, on="post_id", how="left", validate="one_to_one"
    ).rename(columns={"post_id": "id"})
    assert len(submission) == len(test_df) == 1440 and submission["id"].is_unique
    assert submission["label"].notna().all() and submission["label"].isin(labels).all()
    output_path = Path("submission_blend_5fold.csv")
    submission.to_csv(output_path, index=False)
    print(f"Saved {output_path.resolve()}")
    return submission


try:
    blend_submission = export_blend(PREDICTION_DIR)
except FileNotFoundError as exc:
    print(f"Saved predictions are missing: {exc.filename}")
    print("Set LAB2_PREDICTION_DIR to the directory containing the five-fold logits.")

## 4. Competition Results and Error Analysis

`submission_blend_5fold.csv` achieved an OOF macro-F1 of **0.6191**, a Kaggle public score of **0.6427**, and a private score of **0.6289**. The private score was **0.0138** lower than the public score, a drop of 1.38 points on a 0–100 scale.

The public and private scores come from different subsets of the test data. Differences in class counts, wording, or example difficulty could change per-class precision and recall. Macro-F1 weights each class equally, so a small number of different predictions in a less frequent class can have an outsized effect. The blend had an OOF F1 of about 0.4555 for `disgust`, compared with about 0.7792 for `joy`, which makes uneven class difficulty worth investigating.

The blend weight was selected and scored on the same OOF predictions, which may make the local improvement optimistic. Comparing several submissions on the public split can also favor a configuration that happens to score well there. The exact-match rule may cover different fractions of the two leaderboard subsets, and the grouped OOF score does not measure this postprocessing step. These are possible explanations. Establishing their relative importance would require split-level labels or per-class results.